In [1]:
import matplotlib.pyplot as plt
import os
import glob
import pandas_datareader.data as web
import requests
import datetime
import pandas as pd
from dbnomics import fetch_series
from utils import fetch_imf_data, fetch_imf_bulk, fetch_imf_api, fetch_imts, fetch_bop_services, _parse_imf_sdmx_json
from config import country_dict, us_economy_metrics, country_dict_3_char, dict_3_char, dict_2_char


In [2]:
# Paths
imf_cpi_path = "data/imf_cpi.csv"
imf_gdp_path = "data/imf_gdp.csv"
imf_unemployment_path = "data/imf_unemployment_rate.csv"
imf_industrial_production_index_path = "data/imf_industrial_production_index.csv"
imf_exports_path = "data/imf_exports.csv"
imf_imports_path = "data/imf_imports.csv"
imf_services_exports_path = "data/imf_services_exports.csv"
imf_services_imports_path = "data/imf_services_imports.csv"
us_economy_path = "data/us_economy.csv"
cb_policy_rates_path = "data/cb_policy_rates.csv"
global_manufacturing_PMI_path = "data/global_manufacturing_PMI.csv"
im_ex_path = "data/im_ex/"
im_path = "data/im/"
ex_path = "data/ex/"
retail_sales_path = "data/retail_sales.csv"
country_mapping_path = "data/d_country.csv"
exchange_rate_path = "data/exchange_rate.csv"

target_countries = ["VN", "TH", "SG", "ID", "MY", "PH", "BN", "KH", "LA", "MM", "TL", "DE", "FR", "IT", "ES", "NL", "GB", "BE", "AT", "PT", "GR", "FI", "IE", "DK", "SE", "KW", "OM", "BH"]


In [3]:
# Tạo DataFrame Mapping
df_map_2_char = pd.DataFrame(list(dict_2_char.items()), columns=['iso2_code', 'country_name_2c'])
df_map_3_char = pd.DataFrame(list(dict_3_char.items()), columns=['iso3_code', 'country_name'])

# Ghép theo index để tạo bảng mapping 3 cột chuẩn
df_country_mapping = pd.concat([df_map_2_char[['iso2_code']], df_map_3_char], axis=1)

# Reorder cột cho đẹp: iso2_code | iso3_code | country_name
df_country_mapping = df_country_mapping[['iso2_code', 'iso3_code', 'country_name']]

df_country_mapping.to_csv(country_mapping_path)

In [4]:
exchange_rate = fetch_series(
    provider_code="IMF", 
    dataset_code="IFS", # Exchange Rates, US Dollar per Domestic Currency, Period Average, Rate
    max_nb_series = 500000,
    dimensions={
        "FREQ": ["M"],
        # "REF_AREA": list(country_dict.keys())
        "INDICATOR": ['EDNA_USD_XDC_RATE']
    }
)
exchange_rate.to_csv(exchange_rate_path)
exchange_rate_summary = exchange_rate.copy()
exchange_rate_summary['year'] = pd.to_datetime(exchange_rate_summary['period'], errors='coerce').dt.strftime("%Y")
exchange_rate_summary = exchange_rate_summary.groupby(['year', 'REF_AREA'], as_index=False)['value'].mean()

In [17]:
# Pillar 1: System Health & Growth
# 1. REAL GDP GROWTH (Tăng trưởng GDP Thực tế - Hàng Quý 'Q' hoặc Hàng Năm 'A')
# print("Fetching Real GDP Growth Data...")
# imf_gdp = fetch_imf_data(
#     country_dict=country_dict, 
#     dataset_code = "IFS",
#     # metric_suffix="NGDP_R_XDC", #  Real
#     metric_suffix="NGDP_​XDC", # Nominal
#     frequency="A"
# )
imf_gdp = fetch_series(
    provider_code="IMF", 
    dataset_code="IFS", 
    max_nb_series=50000000000,
    dimensions={
        "FREQ": ["A"],
        "INDICATOR": ['NGDP_XDC']
    }
).dropna(subset=['value'])

# Add exchange rate
imf_gdp = imf_gdp.merge(exchange_rate_summary, left_on=['REF_AREA', 'original_period'], right_on=['REF_AREA', 'year'], how='left')
imf_gdp = imf_gdp.rename(columns={
        "value_x": "value",
        'value_y': 'usd/local'
})
imf_gdp = imf_gdp[['@frequency', 'provider_code', 'dataset_code', 'dataset_name',
       'series_code', 'series_name', 'original_period', 'period',
       'original_value', 'value', 'FREQ', 'REF_AREA', 'INDICATOR', 'Frequency',
       'Reference Area', 'Indicator', "usd/local"]]

imf_gdp['local/usd'] = 1/imf_gdp['usd/local']
imf_gdp['value_usd'] = imf_gdp['value']*imf_gdp['usd/local']

# Remove Macao since it's not a country
imf_gdp = imf_gdp[imf_gdp['REF_AREA']!='MO']
imf_gdp.to_csv(imf_gdp_path, index=False)

In [6]:
# Pillar 1: System Health & Growth
# 2. INDUSTRIAL PRODUCTION INDEX - IPI (Thay đổi dataset_code="IFS")
ipi_raw = fetch_imf_bulk(
    dataset_code="IFS",
    metric_code="AIP_IX", 
    frequency="M", 
    country_list=target_countries
)
if not ipi_raw.empty:
    ipi_raw.dropna(subset=['value']).to_csv(imf_industrial_production_index_path, index=False)

--- Đang tải dữ liệu cho mã: AIP_IX (M) ---
✅ Tải thành công 11153 dòng dữ liệu!


In [7]:
# Pillar 2: Price Stability & Monetary Policy
# 3. CPI
imf_cpi = fetch_imf_data(country_dict=country_dict, dataset_code = "CPI", metric_suffix="PCPI_IX", frequency="M").dropna(subset=['value'])
# Show 10 samples
imf_cpi.head(10)
# Export data
imf_cpi.to_csv(imf_cpi_path)

Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.VA.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.TM.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.TV.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.VU.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.SUH.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.U2.PCPI_IX'}


In [8]:
# Pillar 2: Price Stability & Monetary Policy
# 4. Central bank policy rates
cb_policy_rates = fetch_series(
    provider_code="BIS", 
    dataset_code="WS_CBPOL", 
    dimensions={
        "FREQ": ["D"],
        "REF_AREA": list(country_dict.keys())
    }
)
cb_policy_rates.to_csv(cb_policy_rates_path)

In [9]:
# Pillar 3: Labor Market & Consumer Market
# 5. Unemployment Rate
unemployment_raw = fetch_series(
    provider_code="IMF", 
    dataset_code="WEO:latest", 
    dimensions={
        "weo-country": list(country_dict_3_char.keys()),
        "weo-subject": ["LUR"],
        "unit": ["pcent_total_labor_force"]
    }
)
unemployment_raw = unemployment_raw[unemployment_raw['period'] <= pd.Timestamp.today()]
unemployment_raw.to_csv(imf_unemployment_path)

In [10]:
# Pillar 3: Labor Market & Consumer Market
# 6. Retail Sales Growth
# DBnomics: OECD (KEI).
retail_sales_raw_set_1 = fetch_series(
    provider_code="OECD", 
    dataset_code="MEI", 
    dimensions={
        "SUBJECT": ["SLRTTO02"],     #  Sales > Retail trade > Total retail trade > Value
        "MEASURE": ["IXEB"],    # US Dollars, monthly level
        "FREQUENCY": ["M"],     # Lấy theo Tháng (Monthly) để monitor cực nhạy
        "LOCATION": ['AUT', 'BEL', 'BGR', 'CHE', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LTU', 'NLD', 'NOR', 'PRT', 'SVK', 'TUR']
    }
)
retail_sales_raw_set_2 = fetch_series(
    provider_code="OECD", 
    dataset_code="MEI", 
    dimensions={
        "SUBJECT": ["SLRTTO02"],
        "MEASURE": ["ML"],
        "FREQUENCY": ["M"],
        "LOCATION": ['CHN', 'RUS', 'USA', 'ZAF']
    }
)
retail_sales_raw = pd.concat([retail_sales_raw_set_1, retail_sales_raw_set_2], ignore_index=True)
retail_sales_raw.to_csv(retail_sales_path, encoding="utf-8-sig")

In [11]:
# Pillar 4: Trade & Geopolitics
# 7. Global Manufacturing PMI
# DBnomics: WB (World Bank - Pink Sheet Commodity Prices) hoặc IMF.
global_manufacturing_PMI = fetch_series(
    provider_code="OECD", 
    dataset_code="MEI", 
    dimensions={
        "SUBJECT": ["BSCICP02"], 
        "FREQUENCY": ["M"],
    }
)
global_manufacturing_PMI.to_csv(global_manufacturing_PMI_path)

In [12]:
# Pillar 4: Trade & Geopolitics
# 8.1. Merchandise Import Value
imports_data_1 = fetch_imts("MG_FOB_USD", start_period="2010-01", end_period="2026-03")
imports_data_1['period'] = pd.to_datetime(imports_data_1['period'], format="%Y-M%m").dt.strftime('%Y-%m')
import_data_countries = list(imports_data_1['COUNTRY'].unique())
print(f'fetching the first imports_data for countries: {import_data_countries}')

imports_data_2 = fetch_imts("MG_CIF_USD", start_period="2010-01", end_period="2026-03")
imports_data_2 = imports_data_2[~imports_data_2['COUNTRY'].isin(import_data_countries)]
imports_data_2['period'] = pd.to_datetime(imports_data_2['period'], format="%Y-M%m").dt.strftime('%Y-%m')
print(f"fetching the second imports_data for countries: {imports_data_2['COUNTRY'].unique()}")

imports_data = pd.concat([imports_data_1, imports_data_2], ignore_index=True)
imports_data.to_csv(imf_imports_path, index=False, encoding="utf-8-sig")


fetching the first imports_data for countries: ['AUS', 'BMU', 'BRA', 'CAN', 'DOM', 'MEX', 'PER', 'PNG', 'PRY', 'SLB', 'ZAF', 'ZWE']
fetching the second imports_data for countries: <StringArray>
['ABW', 'AFG', 'AGO', 'AIA', 'ALB', 'ANT', 'ARE', 'ARG', 'ARM', 'ASM',
 ...
 'UZB', 'VAT', 'VCT', 'VEN', 'VNM', 'VUT', 'WBG', 'WSM', 'YEM', 'ZMB']
Length: 216, dtype: str


In [13]:
# Pillar 4: Trade & Geopolitics
# 8.2. Merchandise Export Value
export_data = fetch_imts("XG_FOB_USD", start_period="2010-01", end_period="2026-03")
export_data['period'] = pd.to_datetime(export_data['period'], format="%Y-M%m").dt.strftime('%Y-%m')
export_data.head(10)
export_data.to_csv(imf_exports_path, index=False, encoding="utf-8-sig")


In [14]:
# Pillar 4: Trade & Geopolitics
# 8.3. Trade in Services (Exports/Imports Value)
services_target = list(country_dict_3_char.keys())

services_exports = fetch_bop_services("CD_T", services_target, start_period="2010-01", end_period="2026-03")
services_exports['period'] = pd.to_datetime(services_exports['period'])
services_exports['value'] = services_exports['value'] / 3
services_exports = services_exports.loc[services_exports.index.repeat(3)].reset_index(drop=True)
services_exports['period'] = (pd.PeriodIndex(services_exports['period'], freq='M') + services_exports.index % 3).to_timestamp()
services_exports.to_csv(imf_services_exports_path, index=False, encoding="utf-8-sig")

services_imports = fetch_bop_services("DB_T", services_target, start_period="2010-01", end_period="2026-03")
services_imports['period'] = pd.to_datetime(services_imports['period'])
services_imports['value'] = services_imports['value'] / 3
services_imports = services_imports.loc[services_imports.index.repeat(3)].reset_index(drop=True)
services_imports['period'] = (pd.PeriodIndex(services_imports['period'], freq='M') + services_imports.index % 3).to_timestamp()
services_imports.to_csv(imf_services_imports_path, index=False, encoding="utf-8-sig")


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_30032\4226738584.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  services_exports['period'] = pd.to_datetime(services_exports['period'])
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_30032\4226738584.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  services_imports['period'] = pd.to_datetime(services_imports['period'])


In [15]:
# Pillar 4: Trade & Geopolitics
# 9. Commodity Price Index
# DBnomics: WB (World Bank - Pink Sheet Commodity Prices) hoặc IMF.

In [16]:
# Timeframe covering Trump baseline through the entirety of the Biden administration
start = datetime.datetime(2000, 1, 1)
end = datetime.datetime(2025, 12, 31)

# Define the exact FRED series codes mapped to your portfolio naming convention
try:
    df_list = []
    for name, code in us_economy_metrics.items():
        print(f"-> Fetching {name} ({code})...")
        # Pull data directly from St. Louis Fed servers
        df = web.DataReader(code, 'fred', start, end)
        # 1. Clean the individual metric's index and rename columns
        df = df.reset_index().rename(columns={'DATE': 'Date', code: 'Value'})
        # 2. Handle frequency alignment: Forward-fill quarterly data before stacking
        # This ensures April and May inherit Q1 data before it gets mixed with monthly metrics
        if name in ['Real_GDP', 'Manufacturing_Investment']:
            # Create a continuous monthly date range to map the quarterly data onto
            monthly_range = pd.date_range(start=start, end=end, freq='MS')
            df = df.set_index('Date').reindex(monthly_range).ffill().reset_index().rename(columns={'index': 'Date'})
        # 3. Add the metadata column so we know which metric this row belongs to
        df['Metric_Name'] = name
        # Reorder columns to look clean: Date | Metric_Name | Value
        df = df[['Date', 'Metric_Name', 'Value']]        
        df_list.append(df)
    print("Consolidating and formatting data frequencies...")
    # Concatenate all tables along the Date axis
    bi_dataset = pd.concat(df_list, axis=0, ignore_index=True)
    # Forward-fill (ffill) the quarterly data (GDP & Investment) so monthly rows aren't blank
    # This prevents relationship errors inside Power BI's model
    bi_dataset = bi_dataset.ffill()
    # Reset index to turn the Date from an index into a normal clean column
    bi_dataset = bi_dataset.reset_index().rename(columns={'DATE': 'Date'})
    # Export to local project directory
    bi_dataset.to_csv(us_economy_path, index=False)
    print(f"Success! Master file saved as '{us_economy_path}'")
    print(bi_dataset.tail(10))

except Exception as e:
    print(f"Pipeline Execution Failed: {e}")

-> Fetching CPI_All_Items (CPIAUCSL)...
-> Fetching Unemployment_Rate (UNRATE)...
-> Fetching Total_Nonfarm_Payrolls (PAYEMS)...
-> Fetching Real_GDP (GDPC1)...
-> Fetching Manufacturing_Investment (C307RX1Q020SBEA)...
Consolidating and formatting data frequencies...
Success! Master file saved as 'data/us_economy.csv'
      index       Date               Metric_Name    Value
1550   1550 2025-03-01  Manufacturing_Investment  145.228
1551   1551 2025-04-01  Manufacturing_Investment  142.245
1552   1552 2025-05-01  Manufacturing_Investment  142.245
1553   1553 2025-06-01  Manufacturing_Investment  142.245
1554   1554 2025-07-01  Manufacturing_Investment  136.654
1555   1555 2025-08-01  Manufacturing_Investment  136.654
1556   1556 2025-09-01  Manufacturing_Investment  136.654
1557   1557 2025-10-01  Manufacturing_Investment  126.551
1558   1558 2025-11-01  Manufacturing_Investment  126.551
1559   1559 2025-12-01  Manufacturing_Investment  126.551
